# 01 — QLoRA SFT: `google/gemma-4-E4B` on `no_robots`

Run **00_setup_colab.ipynb** first in this same session.

This fine-tunes the Gemma 4 E4B base model with a rank-16 QLoRA adapter on `HuggingFaceH4/no_robots` — deliberately *not* the `FineTome-100k` dataset used in the reference Unsloth recipe this config was benchmarked against (see `data/README.md`). Fits a free Colab T4 (~10GB VRAM).

## Load config

In [ ]:
from slm_prod.utils import load_config

model_cfg = load_config("model.yaml")
lora_cfg = load_config("lora.yaml")
sft_cfg = load_config("sft.yaml")
model_cfg, lora_cfg, sft_cfg


## Load base model in 4-bit (Unsloth)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_cfg["base_model_id"],
    max_seq_length=model_cfg["max_seq_length"],
    dtype=None,
    load_in_4bit=model_cfg["load_in_4bit"],
)


## Qualitative baseline

Generate from the *un-tuned* base model so we have a concrete before/after later.

In [ ]:
FastLanguageModel.for_inference(model)

prompt = "Explain what a GPTQ quantized model is, in two sentences, to a non-technical reader."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=120)
baseline_generation = tokenizer.decode(out[0], skip_special_tokens=True)
print(baseline_generation)


## Attach LoRA adapters

In [ ]:
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    use_gradient_checkpointing=lora_cfg["use_gradient_checkpointing"],
    random_state=lora_cfg["random_state"],
)
model.print_trainable_parameters()


## Load and format `no_robots`

In [ ]:
from slm_prod.data import load_sft_dataset

train_dataset = load_sft_dataset(tokenizer, split=sft_cfg["dataset_split"])
eval_dataset = load_sft_dataset(tokenizer, split=sft_cfg["eval_split"])
print(train_dataset)
print(train_dataset[0]["text"][:500])


## Train

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=sft_cfg["output_dir"],
    per_device_train_batch_size=sft_cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=sft_cfg["gradient_accumulation_steps"],
    num_train_epochs=sft_cfg["num_train_epochs"],
    learning_rate=sft_cfg["learning_rate"],
    lr_scheduler_type=sft_cfg["lr_scheduler_type"],
    warmup_ratio=sft_cfg["warmup_ratio"],
    weight_decay=sft_cfg["weight_decay"],
    optim=sft_cfg["optim"],
    logging_steps=sft_cfg["logging_steps"],
    eval_strategy="steps",
    eval_steps=sft_cfg["eval_steps"],
    save_steps=sft_cfg["save_steps"],
    save_total_limit=sft_cfg["save_total_limit"],
    seed=sft_cfg["seed"],
    dataset_text_field="text",
    max_seq_length=model_cfg["max_seq_length"],
    packing=False,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)
trainer_stats = trainer.train()


## Qualitative check: same prompt, after SFT

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=120)
sft_generation = tokenizer.decode(out[0], skip_special_tokens=True)

print("BASE:\n", baseline_generation, "\n")
print("SFT:\n", sft_generation)


## Save (and optionally push) the adapter

In [ ]:
model.save_pretrained(sft_cfg["adapter_dir"])
tokenizer.save_pretrained(sft_cfg["adapter_dir"])
print(f"Adapter saved to {sft_cfg['adapter_dir']}")

# Optional: push to your own HF account so later notebooks/sessions can pull it
# push_repo_id = "your-username/gemma-4-e4b-no_robots-sft"
# model.push_to_hub(push_repo_id)
# tokenizer.push_to_hub(push_repo_id)


Continue with **02_merge_and_quantize_gptq.ipynb**.